In [1]:
#import plotly.express as px
import numpy as np
import xarray as xr
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
def wrap_lon(ds):
    
    # method to 'wrap' longitudes from (0,360) to (-180,180) & sort into ascending order
    
    if "longitude" in ds.coords:
        lon = "longitude"
        lat = "latitude"
    elif "lon" in ds.coords:
        lon = "lon"
        lat = "lat"
    else: 
        # can only wrap longitude
        return ds
    
    if ds[lon].max() > 180:
        ds[lon] = (ds[lon].dims, (((ds[lon].values + 180) % 360) - 180), ds[lon].attrs)
        
    if lon in ds.dims:
        ds = ds.reindex({ lon : np.sort(ds[lon]) })
        ds = ds.reindex({ lat : np.sort(ds[lat]) })
    return ds

In [3]:
tmax_era5=wrap_lon(xr.open_dataset("/net/pc230042/nobackup/users/sager/nobackup_2_old/ERA5-CX-READY/era5_tmax_daily.nc"))

In [4]:
#select climatology
tmax_era5_clim=tmax_era5.sel(time=slice('1991','2020'))

1 calculate the 30 day running mean 

2 select the stats grouped by day for the climatological period 

x number of days mean around the day d

In [5]:
def sliding_stat_by_dayofyear(data, pad=15, method='std', quantile_val=0.9):
    """
    Compute day-of-year-based sliding window statistics (mean, std, or quantile) across years.

    Parameters:
    -----------
    data : xr.DataArray
        3D DataArray with dimensions ('time', 'lat', 'lon')
    pad : int
        Number of days on either side to include in the window (default: 15 → 30-day window)
    method : str
        Statistic to compute: 'std', 'mean', or 'quantile'
    quantile_val : float
        Quantile to compute if method='quantile' (e.g., 0.9 for 90th percentile)

    Returns:
    --------
    xr.DataArray
        DataArray of shape (dayofyear, lat, lon) with the selected statistic
        Each [d, :, :] slice contains the 30-day std around day d, computed across all years.
    """

    # Sanity check
    if method not in ['std', 'mean', 'quantile']:
        raise ValueError("method must be one of: 'std', 'mean', 'quantile'")

    # Remove Feb 29 to standardize 365-day calendar
    data = data.sel(time=~((data.time.dt.month == 2) & (data.time.dt.day == 29)))

    days = np.arange(1, 366)  # Days of year
    dayofyear = data.time.dt.dayofyear
    result_list = []

    for day in days:
        # Build ±pad-day window (cyclically)
        window_days = [(day + offset - 1) % 365 + 1 for offset in range(-pad, pad + 1)]
        mask = dayofyear.isin(window_days)
        window_data = data.sel(time=mask)

        # Compute selected statistic
        if method == 'std':
            stat = window_data.std(dim='time')
        elif method == 'mean':
            stat = window_data.mean(dim='time')
        elif method == 'quantile':
            stat = window_data.quantile(quantile_val, dim='time')
        
        result_list.append(stat)

    # Combine results
    result = xr.concat(result_list, dim='dayofyear')
    result = result.assign_coords(dayofyear=days)

    return result

In [ ]:
# 15-day window mean
std_result = sliding_stat_by_dayofyear(tmax_era5_clim, pad=15, method='std')

In [6]:
# 15-day window mean
mean_result = sliding_stat_by_dayofyear(tmax_era5_clim, pad=15, method='mean')

In [ ]:

# 90th percentile (quantile) over 31-day window
#q90_result = sliding_stat_by_dayofyear(tmax_era5_clim, pad=15, method='quantile', quantile_val=0.95)

In [8]:
mean_result.to_netcdf('tmax_era5_climatology_30day_window_mean_1991_2020.nc')
std_result.to_netcdf('tmax_era5_climatology_30day_window_std_1991_2020.nc')